# 06. Carga Incremental de Capa Silver — Data Vault 2.0 (13 Tablas)

Lee cronológicamente las particiones Hive de Staging (`data/stg/sa_dataset_obras_publicas/fecha_foto=YYYY-MM-DD/*.parquet`, a partir de `2023-07-11`) y construye las 13 tablas Parquet de la capa Silver en `data/silver/` bajo un modelo Data Vault 2.0 estricto:

- **Hubs (4 — SCD Tipo 1 con `COALESCE(nuevo, actual)`)**:
  1. `hub_obra_publica` (`codigo_infobras` + 7 atributos del dominio `obra`)
  2. `hub_entidad` (`codigo_entidad`, `nivel_gobierno`, `sector_entidad`)
  3. `hub_proyecto` (`codigo_unico`, `tipo_formato`, excluyendo `'0'`/vacíos)
  4. `hub_contratista` (`ruc`, `razon_social_empresa_consorcio`, unificando `ruc_1` y `ruc_2`)
- **Links (3 — SCD Tipo 2 con `start_date`, `end_date`, `fecha_foto`)**:
  5. `link_obra_entidad` (`codigo_infobras` ↔ `codigo_entidad`)
  6. `link_obra_proyecto` (`codigo_infobras` ↔ `codigo_unico`)
  7. `link_obra_contratista` (`(codigo_infobras, rol)` ↔ `ruc`, con `rol IN ('EJECUCION', 'SUPERVISION')`)
- **Satélites (6)**:
  8. `sat_obra_paralizacion` (SCD2: `flg_existe_paralizacion`, `causal_paralizacion`, `categoria_paralizacion`, `fecha_paralizacion`)
  9. `sat_obra_estado_ejecucion` (SCD2: `estado_ejecucion`)
  10. `sat_obra_ubigeo` (SCD1: `departamento`, `provincia`, `distrito`)
  11. `sat_obra_expediente_tecnico` (SCD2: `fecha_aprobacion_expediente`, `monto_aprobacion_et`, `monto_aprobado_soles`)
  12. `sat_proyecto` (SCD2 por `codigo_unico`: `n_obras_relacionadas_proyecto`)
  13. `sat_link_obra_contratista` (SCD2 por `(codigo_infobras, rol, ruc)`: `monto_contrato_soles`)

Reglas funcionales aplicadas: `docs/dataflow/silver_layer_rules.md` (Reglas 1 a 9), `configs/categoria_paralizacion.yaml` y `docs/designs/silver_datavault_y_gold_poc_ds_obras_publicas.md`.

**Modos de ejecución (`MODO`)**:
- `"PRUEBA"`: procesa solo las primeras `N_FOTOS_PRUEBA` fotos desde `FECHA_MINIMA`.
- `"COMPLETA"`: reconstruye desde cero todas las particiones válidas (`fecha_foto >= FECHA_MINIMA`).
- `"INCREMENTAL"`: lee las 13 tablas existentes en `data/silver/`, toma `MAX(fecha_foto)` de `hub_obra_publica` como marca de agua y procesa únicamente las particiones posteriores.

## 1. Parámetros de ejecución

In [ ]:
# ==============================================================================
# PARÁMETROS DE ENTRADA
# ==============================================================================
MODO = "COMPLETA"          # "PRUEBA" | "COMPLETA" | "INCREMENTAL"
N_FOTOS_PRUEBA = 3         # Cantidad de particiones iniciales en MODO = "PRUEBA"
FECHA_MINIMA = "2023-07-11"  # Regla 2: desde esta foto flg_existe_paralizacion es binario (true/false)

COMPRESION = "ZSTD"
COMPRESION_NIVEL = 9
ROW_GROUP_SIZE = 100_000
DETENER_ANTE_ERROR = True

# Rutas (None = rutas por defecto del proyecto)
DIR_ORIGEN = None          # por defecto: data/stg/sa_dataset_obras_publicas
DIR_DESTINO = None         # por defecto: data/silver
DIR_LOGS = None            # por defecto: data/silver/logs
CONFIG_CAT_PATH = None     # por defecto: configs/categoria_paralizacion.yaml
# ==============================================================================

## 2. Configuración, rutas, mapeo YAML y logger de ejecución

In [ ]:
import gc
import getpass
import logging
import platform
import re
import shutil
import socket
import sys
import time
import traceback
from datetime import date, datetime
from pathlib import Path

import duckdb
import pandas as pd
import yaml

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 90)

MODO = str(MODO).upper().strip()
assert MODO in ("PRUEBA", "COMPLETA", "INCREMENTAL"), 'MODO debe ser "PRUEBA", "COMPLETA" o "INCREMENTAL"'


def obtener_raiz_proyecto() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "data" / "stg").exists() or (carpeta / "configs").exists():
            return carpeta
    return actual


PROJECT_ROOT = obtener_raiz_proyecto()
STG_DIR = Path(DIR_ORIGEN) if DIR_ORIGEN else PROJECT_ROOT / "data" / "stg" / "sa_dataset_obras_publicas"
SILVER_DIR = Path(DIR_DESTINO) if DIR_DESTINO else PROJECT_ROOT / "data" / "silver"
WORK_DIR = SILVER_DIR / "_work"
LOGS_DIR = Path(DIR_LOGS) if DIR_LOGS else SILVER_DIR / "logs"
RUTA_CONFIG_CAT = Path(CONFIG_CAT_PATH) if CONFIG_CAT_PATH else PROJECT_ROOT / "configs" / "categoria_paralizacion.yaml"

SILVER_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

assert RUTA_CONFIG_CAT.exists(), f"No existe archivo de configuración de categorías: {RUTA_CONFIG_CAT}"
CFG_CATEGORIA = yaml.safe_load(RUTA_CONFIG_CAT.read_text(encoding="utf-8"))
CATEGORIA_DEFAULT = str(CFG_CATEGORIA["default_category"]).strip()
CATEGORIAS_VALIDAS = [str(c).strip() for c in CFG_CATEGORIA["categories"]]
assert CATEGORIA_DEFAULT in CATEGORIAS_VALIDAS, f"default_category '{CATEGORIA_DEFAULT}' fuera de categories"

MAPEO_CATEGORIA_PARALIZACION: dict[str, str] = {}
for k_raw, v_raw in CFG_CATEGORIA["mapping"].items():
    k_norm = str(k_raw).strip().lower()
    v_norm = str(v_raw).strip()
    assert k_norm, "Clave vacía en mapping de categoria_paralizacion.yaml"
    assert v_norm in CATEGORIAS_VALIDAS, f"Categoría '{v_norm}' fuera de categories para '{k_raw}'"
    assert k_norm not in MAPEO_CATEGORIA_PARALIZACION, f"Clave duplicada al normalizar: '{k_raw}'"
    MAPEO_CATEGORIA_PARALIZACION[k_norm] = v_norm

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = LOGS_DIR / f"carga_silver_{MODO.lower()}_{RUN_ID}.log"
RESUMEN_FOTOS_PATH = LOGS_DIR / f"carga_silver_{MODO.lower()}_{RUN_ID}_fotos.csv"
RESUMEN_TABLAS_PATH = LOGS_DIR / f"carga_silver_{MODO.lower()}_{RUN_ID}_tablas.csv"


def configurar_logger() -> logging.Logger:
    """Configura un logger con salida simultánea a archivo y consola."""
    logger = logging.getLogger("carga_silver")
    for h in list(logger.handlers):
        h.close()
        logger.removeHandler(h)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(LOG_PATH, encoding="utf-8")
    sh = logging.StreamHandler(sys.stdout)
    for h in (fh, sh):
        h.setFormatter(fmt)
        logger.addHandler(h)
    return logger


log = configurar_logger()

log.info("=" * 100)
log.info("INICIO DE CORRIDA SILVER (DATA VAULT 2.0 - 13 TABLAS) | run_id=%s | modo=%s", RUN_ID, MODO)
log.info("Usuario=%s | Host=%s | SO=%s", getpass.getuser(), socket.gethostname(), platform.platform())
log.info("Python=%s | pandas=%s | duckdb=%s", platform.python_version(), pd.__version__, duckdb.__version__)
log.info("Origen Staging : %s", STG_DIR)
log.info("Destino Silver : %s", SILVER_DIR)
log.info("Trabajo (_work): %s", WORK_DIR)
log.info("Config YAML    : %s (%d causales -> %d categorías, default=%s)", RUTA_CONFIG_CAT, len(MAPEO_CATEGORIA_PARALIZACION), len(CATEGORIAS_VALIDAS), CATEGORIA_DEFAULT)
log.info("Log            : %s", LOG_PATH)
log.info(
    "Parámetros: FECHA_MINIMA=%s | N_FOTOS_PRUEBA=%s | COMPRESION=%s(%s) | DETENER_ANTE_ERROR=%s",
    FECHA_MINIMA, N_FOTOS_PRUEBA, COMPRESION, COMPRESION_NIVEL, DETENER_ANTE_ERROR,
)

## 3. Definición declarativa de esquemas (`SCHEMAS_SILVER`, `TABLAS_SCD1`, `TABLAS_SCD2`)

In [ ]:
SCHEMAS_SILVER = {
    "hub_obra_publica": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("modalidad_ejecucion_obra", "VARCHAR"),
            ("naturaleza_obra", "VARCHAR"),
            ("tipo_obra_nivel_1", "VARCHAR"),
            ("tipo_obra_nivel_2", "VARCHAR"),
            ("tipo_obra_nivel_3", "VARCHAR"),
            ("fecha_inicio_obra", "DATE"),
            ("fecha_finalizacion_real", "DATE"),
            ("first_seen_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras"],
    },
    "hub_entidad": {
        "columns": [
            ("codigo_entidad", "VARCHAR"),
            ("nivel_gobierno", "VARCHAR"),
            ("sector_entidad", "VARCHAR"),
            ("first_seen_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_entidad"],
    },
    "hub_proyecto": {
        "columns": [
            ("codigo_unico", "VARCHAR"),
            ("tipo_formato", "VARCHAR"),
            ("first_seen_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_unico"],
    },
    "hub_contratista": {
        "columns": [
            ("ruc", "VARCHAR"),
            ("razon_social_empresa_consorcio", "VARCHAR"),
            ("first_seen_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["ruc"],
    },
    "link_obra_entidad": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("codigo_entidad", "VARCHAR"),
            ("start_date", "DATE"),
            ("end_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras", "start_date"],
    },
    "link_obra_proyecto": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("codigo_unico", "VARCHAR"),
            ("start_date", "DATE"),
            ("end_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras", "start_date"],
    },
    "link_obra_contratista": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("rol", "VARCHAR"),
            ("ruc", "VARCHAR"),
            ("start_date", "DATE"),
            ("end_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras", "rol", "start_date"],
    },
    "sat_obra_paralizacion": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("flg_existe_paralizacion", "BOOLEAN"),
            ("causal_paralizacion", "VARCHAR"),
            ("categoria_paralizacion", "VARCHAR"),
            ("fecha_paralizacion", "DATE"),
            ("start_date", "DATE"),
            ("end_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras", "start_date"],
    },
    "sat_obra_estado_ejecucion": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("estado_ejecucion", "VARCHAR"),
            ("start_date", "DATE"),
            ("end_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras", "start_date"],
    },
    "sat_obra_ubigeo": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("departamento", "VARCHAR"),
            ("provincia", "VARCHAR"),
            ("distrito", "VARCHAR"),
            ("first_seen_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras"],
    },
    "sat_obra_expediente_tecnico": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("fecha_aprobacion_expediente", "DATE"),
            ("monto_aprobacion_et", "DOUBLE"),
            ("monto_aprobado_soles", "DOUBLE"),
            ("start_date", "DATE"),
            ("end_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras", "start_date"],
    },
    "sat_proyecto": {
        "columns": [
            ("codigo_unico", "VARCHAR"),
            ("n_obras_relacionadas_proyecto", "INTEGER"),
            ("start_date", "DATE"),
            ("end_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_unico", "start_date"],
    },
    "sat_link_obra_contratista": {
        "columns": [
            ("codigo_infobras", "VARCHAR"),
            ("rol", "VARCHAR"),
            ("ruc", "VARCHAR"),
            ("monto_contrato_soles", "DOUBLE"),
            ("start_date", "DATE"),
            ("end_date", "DATE"),
            ("fecha_foto", "DATE"),
        ],
        "order_by": ["codigo_infobras", "rol", "ruc", "start_date"],
    },
}

TABLAS_SCD1 = [
    {
        "nombre": "hub_obra_publica",
        "key_col": "codigo_infobras",
        "attr_cols": [
            "modalidad_ejecucion_obra",
            "naturaleza_obra",
            "tipo_obra_nivel_1",
            "tipo_obra_nivel_2",
            "tipo_obra_nivel_3",
            "fecha_inicio_obra",
            "fecha_finalizacion_real",
        ],
        "fuente_sql": """
            SELECT
                codigo_infobras,
                modalidad_ejecucion_obra,
                naturaleza_obra,
                tipo_obra_nivel_1,
                tipo_obra_nivel_2,
                tipo_obra_nivel_3,
                fecha_inicio_obra,
                fecha_finalizacion_real
            FROM foto_actual
        """,
        "tipo": "HUB",
    },
    {
        "nombre": "hub_entidad",
        "key_col": "codigo_entidad",
        "attr_cols": ["nivel_gobierno", "sector_entidad"],
        "fuente_sql": """
            WITH base AS (
                SELECT trim(codigo_entidad) AS codigo_entidad, nivel_gobierno, sector_entidad
                FROM foto_actual
                WHERE codigo_entidad IS NOT NULL AND trim(codigo_entidad) != ''
            ),
            claves AS (
                SELECT DISTINCT codigo_entidad FROM base
            ),
            ng AS (
                SELECT codigo_entidad, nivel_gobierno, count(*) AS cnt
                FROM base
                WHERE nivel_gobierno IS NOT NULL
                GROUP BY codigo_entidad, nivel_gobierno
                QUALIFY ROW_NUMBER() OVER (PARTITION BY codigo_entidad ORDER BY cnt DESC, nivel_gobierno ASC) = 1
            ),
            se AS (
                SELECT codigo_entidad, sector_entidad, count(*) AS cnt
                FROM base
                WHERE sector_entidad IS NOT NULL
                GROUP BY codigo_entidad, sector_entidad
                QUALIFY ROW_NUMBER() OVER (PARTITION BY codigo_entidad ORDER BY cnt DESC, sector_entidad ASC) = 1
            )
            SELECT c.codigo_entidad, ng.nivel_gobierno, se.sector_entidad
            FROM claves c
            LEFT JOIN ng USING (codigo_entidad)
            LEFT JOIN se USING (codigo_entidad)
        """,
        "tipo": "HUB",
    },
    {
        "nombre": "hub_proyecto",
        "key_col": "codigo_unico",
        "attr_cols": ["tipo_formato"],
        "fuente_sql": """
            WITH base AS (
                SELECT trim(codigo_unico) AS codigo_unico, tipo_formato
                FROM foto_actual
                WHERE codigo_unico IS NOT NULL AND trim(codigo_unico) NOT IN ('', '0')
            ),
            claves AS (
                SELECT DISTINCT codigo_unico FROM base
            ),
            tf AS (
                SELECT codigo_unico, tipo_formato, count(*) AS cnt
                FROM base
                WHERE tipo_formato IS NOT NULL
                GROUP BY codigo_unico, tipo_formato
                QUALIFY ROW_NUMBER() OVER (PARTITION BY codigo_unico ORDER BY cnt DESC, tipo_formato ASC) = 1
            )
            SELECT c.codigo_unico, tf.tipo_formato
            FROM claves c
            LEFT JOIN tf USING (codigo_unico)
        """,
        "tipo": "HUB",
    },
    {
        "nombre": "hub_contratista",
        "key_col": "ruc",
        "attr_cols": ["razon_social_empresa_consorcio"],
        "fuente_sql": """
            WITH u AS (
                SELECT trim(ruc_1) AS ruc, razon_social_empresa_consorcio_1 AS rs
                FROM foto_actual
                WHERE ruc_1 IS NOT NULL AND trim(ruc_1) != ''
                UNION ALL
                SELECT trim(ruc_2) AS ruc, razon_social_empresa_consorcio_2 AS rs
                FROM foto_actual
                WHERE ruc_2 IS NOT NULL AND trim(ruc_2) != ''
            ),
            rucs AS (
                SELECT DISTINCT ruc FROM u
            ),
            nombres AS (
                SELECT ruc, rs, count(*) AS cnt
                FROM u
                WHERE rs IS NOT NULL
                GROUP BY ruc, rs
                QUALIFY ROW_NUMBER() OVER (PARTITION BY ruc ORDER BY cnt DESC, rs ASC) = 1
            )
            SELECT r.ruc, n.rs AS razon_social_empresa_consorcio
            FROM rucs r
            LEFT JOIN nombres n USING (ruc)
        """,
        "tipo": "HUB",
    },
    {
        "nombre": "sat_obra_ubigeo",
        "key_col": "codigo_infobras",
        "attr_cols": ["departamento", "provincia", "distrito"],
        "fuente_sql": """
            SELECT codigo_infobras, departamento, provincia, distrito
            FROM foto_actual
        """,
        "tipo": "SAT_SCD1",
    },
]

TABLAS_SCD2 = [
    {
        "nombre": "link_obra_entidad",
        "key_cols": ["codigo_infobras"],
        "cols_track": ["codigo_entidad"],
        "fuente_sql": """
            SELECT codigo_infobras, trim(codigo_entidad) AS codigo_entidad
            FROM foto_actual
            WHERE codigo_entidad IS NOT NULL AND trim(codigo_entidad) != ''
        """,
        "tipo": "LINK",
    },
    {
        "nombre": "link_obra_proyecto",
        "key_cols": ["codigo_infobras"],
        "cols_track": ["codigo_unico"],
        "fuente_sql": """
            SELECT codigo_infobras, trim(codigo_unico) AS codigo_unico
            FROM foto_actual
            WHERE codigo_unico IS NOT NULL AND trim(codigo_unico) NOT IN ('', '0')
        """,
        "tipo": "LINK",
    },
    {
        "nombre": "link_obra_contratista",
        "key_cols": ["codigo_infobras", "rol"],
        "cols_track": ["ruc"],
        "fuente_sql": """
            SELECT codigo_infobras, 'EJECUCION' AS rol, trim(ruc_1) AS ruc
            FROM foto_actual
            WHERE ruc_1 IS NOT NULL AND trim(ruc_1) != ''
            UNION ALL
            SELECT codigo_infobras, 'SUPERVISION' AS rol, trim(ruc_2) AS ruc
            FROM foto_actual
            WHERE ruc_2 IS NOT NULL AND trim(ruc_2) != ''
        """,
        "tipo": "LINK",
    },
    {
        "nombre": "sat_obra_paralizacion",
        "key_cols": ["codigo_infobras"],
        "cols_track": [
            "flg_existe_paralizacion",
            "causal_paralizacion",
            "categoria_paralizacion",
            "fecha_paralizacion",
        ],
        "fuente_sql": """
            SELECT
                codigo_infobras,
                flg_existe_paralizacion,
                causal_paralizacion,
                categoria_paralizacion,
                fecha_paralizacion
            FROM foto_actual
            WHERE flg_existe_paralizacion IS NOT NULL
        """,
        "tipo": "SAT",
    },
    {
        "nombre": "sat_obra_estado_ejecucion",
        "key_cols": ["codigo_infobras"],
        "cols_track": ["estado_ejecucion"],
        "fuente_sql": """
            SELECT codigo_infobras, estado_ejecucion
            FROM foto_actual
            WHERE estado_ejecucion IS NOT NULL
        """,
        "tipo": "SAT",
    },
    {
        "nombre": "sat_obra_expediente_tecnico",
        "key_cols": ["codigo_infobras"],
        "cols_track": [
            "fecha_aprobacion_expediente",
            "monto_aprobacion_et",
            "monto_aprobado_soles",
        ],
        "fuente_sql": """
            SELECT
                codigo_infobras,
                fecha_aprobacion_expediente,
                monto_aprobacion_et,
                monto_aprobado_soles
            FROM foto_actual
            WHERE fecha_aprobacion_expediente IS NOT NULL
               OR monto_aprobacion_et IS NOT NULL
               OR monto_aprobado_soles IS NOT NULL
        """,
        "tipo": "SAT",
    },
    {
        "nombre": "sat_proyecto",
        "key_cols": ["codigo_unico"],
        "cols_track": ["n_obras_relacionadas_proyecto"],
        "fuente_sql": """
            SELECT
                trim(codigo_unico) AS codigo_unico,
                MAX(n_obras_relacionadas_proyecto) AS n_obras_relacionadas_proyecto
            FROM foto_actual
            WHERE codigo_unico IS NOT NULL
              AND trim(codigo_unico) NOT IN ('', '0')
              AND n_obras_relacionadas_proyecto IS NOT NULL
            GROUP BY trim(codigo_unico)
        """,
        "tipo": "SAT",
    },
    {
        "nombre": "sat_link_obra_contratista",
        "key_cols": ["codigo_infobras", "rol", "ruc"],
        "cols_track": ["monto_contrato_soles"],
        "fuente_sql": """
            SELECT
                codigo_infobras,
                'EJECUCION' AS rol,
                trim(ruc_1) AS ruc,
                monto_contrato_soles_1 AS monto_contrato_soles
            FROM foto_actual
            WHERE ruc_1 IS NOT NULL AND trim(ruc_1) != '' AND monto_contrato_soles_1 IS NOT NULL
            UNION ALL
            SELECT
                codigo_infobras,
                'SUPERVISION' AS rol,
                trim(ruc_2) AS ruc,
                monto_contrato_soles_2 AS monto_contrato_soles
            FROM foto_actual
            WHERE ruc_2 IS NOT NULL AND trim(ruc_2) != '' AND monto_contrato_soles_2 IS NOT NULL
        """,
        "tipo": "SAT_LINK",
    },
]

SCD1_NOMBRES = {s["nombre"] for s in TABLAS_SCD1}
assert len(SCHEMAS_SILVER) == 13
assert len(TABLAS_SCD1) + len(TABLAS_SCD2) == 13

pd.DataFrame(
    [
        {
            "tabla": t,
            "modelo": "SCD1" if t in SCD1_NOMBRES else "SCD2",
            "columnas": len(spec["columns"]),
            "order_by": ", ".join(spec["order_by"]),
        }
        for t, spec in SCHEMAS_SILVER.items()
    ]
)

## 4. Descubrimiento de particiones, registro de catálogo y marca de agua

In [ ]:
PATRON_PARTICION = re.compile(r"^fecha_foto=(\d{4}-\d{2}-\d{2})$")


def registrar_mapa_categoria_duckdb(con: duckdb.DuckDBPyConnection) -> None:
    """Registra la tabla temporal _map_categoria_paralizacion desde la variable de configuración YAML."""
    con.execute("""
        CREATE OR REPLACE TEMP TABLE _map_categoria_paralizacion (
            causal_norm VARCHAR,
            categoria_paralizacion VARCHAR
        )
    """)
    filas = list(MAPEO_CATEGORIA_PARALIZACION.items())
    if filas:
        con.executemany("INSERT INTO _map_categoria_paralizacion VALUES (?, ?)", filas)


def listar_particiones_stg(fecha_minima: str = FECHA_MINIMA) -> list[tuple[date, Path]]:
    """Lista las particiones Hive válidas en Staging ordenadas ascendentemente."""
    assert STG_DIR.exists(), f"No existe el directorio Staging: {STG_DIR}"
    f_min = date.fromisoformat(str(fecha_minima))
    particiones = []
    for carpeta in STG_DIR.iterdir():
        if not carpeta.is_dir():
            continue
        m = PATRON_PARTICION.match(carpeta.name)
        if not m:
            continue
        f_part = date.fromisoformat(m.group(1))
        if f_part < f_min:
            log.info("Partición ignorada por Regla 2 (< %s): %s", f_min, carpeta.name)
            continue
        if not list(carpeta.glob("*.parquet")):
            log.warning("Partición vacía ignorada (sin *.parquet): %s", carpeta.name)
            continue
        particiones.append((f_part, carpeta))
    particiones.sort(key=lambda x: x[0])
    return particiones


def inicializar_estado_duckdb(
    con: duckdb.DuckDBPyConnection,
    modo: str,
    particiones: list[tuple[date, Path]],
) -> tuple[list[tuple[date, Path]], date | None]:
    """Inicializa las 13 tablas en DuckDB y retorna (particiones_pendientes, watermark)."""
    if modo in ("COMPLETA", "PRUEBA"):
        for tabla, spec in SCHEMAS_SILVER.items():
            ddl_cols = ", ".join(f'"{c}" {t}' for c, t in spec["columns"])
            con.execute(f'CREATE OR REPLACE TABLE "{tabla}" ({ddl_cols})')
        pendientes = particiones[: int(N_FOTOS_PRUEBA)] if modo == "PRUEBA" else particiones
        return pendientes, None

    # MODO == "INCREMENTAL": cargar estado previo de las 13 tablas desde SILVER_DIR
    watermarks = {}
    for tabla in SCHEMAS_SILVER:
        ruta_pq = SILVER_DIR / tabla / "part-00000.parquet"
        if not ruta_pq.exists():
            raise FileNotFoundError(
                f"En MODO='INCREMENTAL' deben existir las 13 tablas en {SILVER_DIR}. Falta: {ruta_pq}"
            )
        con.execute(f"CREATE OR REPLACE TABLE \"{tabla}\" AS SELECT * FROM read_parquet('{ruta_pq.as_posix()}')")
        wm = con.execute(f'SELECT MAX(fecha_foto) FROM "{tabla}"').fetchall()[0][0]
        watermarks[tabla] = wm
        n_rows = con.execute(f'SELECT count(*) FROM "{tabla}"').fetchall()[0][0]
        log.info("[INCREMENTAL] Tabla base %-28s cargada: %10s filas | max(fecha_foto)=%s", tabla, f"{n_rows:,}", wm)

    watermark = watermarks["hub_obra_publica"]
    if watermark is None:
        raise ValueError("hub_obra_publica está vacía; ejecute primero MODO='COMPLETA'.")
    incoherentes = {t: wm for t, wm in watermarks.items() if wm is not None and wm > watermark}
    assert not incoherentes, f"Watermark inconsistente (tablas adelantadas respecto al Hub {watermark}): {incoherentes}"

    pendientes = [(f, p) for f, p in particiones if f > watermark]
    return pendientes, watermark

## 5. Preprocesamiento por foto (Regla 1, Regla 3 y Regla 8)

In [ ]:
def cargar_foto_limpia(con: duckdb.DuckDBPyConnection, fecha_foto: date, ruta_particion: Path) -> dict:
    """
    Lee una partición de Staging (33 columnas proyectadas) en TEMP TABLE foto_actual aplicando:
    - Regla 1: deduplicación por codigo_infobras con fecha_registro_avance más reciente.
    - Regla 3: normalización de causal_paralizacion y fecha_paralizacion a NULL si flg_existe_paralizacion = FALSE.
    - Regla 8: resolución de categoria_paralizacion cruzando contra _map_categoria_paralizacion (YAML).
    """
    glob_pq = (ruta_particion / "*.parquet").as_posix()
    cat_default_sql = CATEGORIA_DEFAULT.replace("'", "''")

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE foto_bruta AS
        SELECT
            CAST(codigo_infobras AS VARCHAR) AS codigo_infobras,
            CAST(modalidad_ejecucion_obra AS VARCHAR) AS modalidad_ejecucion_obra,
            CAST(naturaleza_obra AS VARCHAR) AS naturaleza_obra,
            CAST(tipo_obra_nivel_1 AS VARCHAR) AS tipo_obra_nivel_1,
            CAST(tipo_obra_nivel_2 AS VARCHAR) AS tipo_obra_nivel_2,
            CAST(tipo_obra_nivel_3 AS VARCHAR) AS tipo_obra_nivel_3,
            CAST(fecha_inicio_obra AS DATE) AS fecha_inicio_obra,
            CAST(fecha_finalizacion_real AS DATE) AS fecha_finalizacion_real,
            CAST(codigo_entidad AS VARCHAR) AS codigo_entidad,
            CAST(nivel_gobierno AS VARCHAR) AS nivel_gobierno,
            CAST(sector_entidad AS VARCHAR) AS sector_entidad,
            CAST(codigo_unico AS VARCHAR) AS codigo_unico,
            CAST(tipo_formato AS VARCHAR) AS tipo_formato,
            CAST(n_obras_relacionadas_proyecto AS INTEGER) AS n_obras_relacionadas_proyecto,
            CAST(departamento AS VARCHAR) AS departamento,
            CAST(provincia AS VARCHAR) AS provincia,
            CAST(distrito AS VARCHAR) AS distrito,
            CAST(fecha_aprobacion_expediente AS DATE) AS fecha_aprobacion_expediente,
            CAST(monto_aprobacion_et AS DOUBLE) AS monto_aprobacion_et,
            CAST(monto_aprobado_soles AS DOUBLE) AS monto_aprobado_soles,
            CAST(ruc_1 AS VARCHAR) AS ruc_1,
            CAST(razon_social_empresa_consorcio_1 AS VARCHAR) AS razon_social_empresa_consorcio_1,
            CAST(monto_contrato_soles_1 AS DOUBLE) AS monto_contrato_soles_1,
            CAST(ruc_2 AS VARCHAR) AS ruc_2,
            CAST(razon_social_empresa_consorcio_2 AS VARCHAR) AS razon_social_empresa_consorcio_2,
            CAST(monto_contrato_soles_2 AS DOUBLE) AS monto_contrato_soles_2,
            CAST(flg_existe_paralizacion AS BOOLEAN) AS flg_existe_paralizacion,
            CAST(causal_paralizacion AS VARCHAR) AS causal_paralizacion,
            CAST(fecha_paralizacion AS DATE) AS fecha_paralizacion,
            CAST(estado_ejecucion AS VARCHAR) AS estado_ejecucion,
            CAST(fecha_registro_avance AS DATE) AS fecha_registro_avance,
            CAST(ano_avance AS INTEGER) AS ano_avance,
            CAST(mes_avance AS INTEGER) AS mes_avance
        FROM read_parquet('{glob_pq}', hive_partitioning = false)
        WHERE codigo_infobras IS NOT NULL AND trim(codigo_infobras) != ''
    """)

    filas_brutas = int(con.execute("SELECT count(*) FROM foto_bruta").fetchall()[0][0])

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE foto_actual AS
        WITH dedup AS (
            SELECT
                codigo_infobras,
                modalidad_ejecucion_obra,
                naturaleza_obra,
                tipo_obra_nivel_1,
                tipo_obra_nivel_2,
                tipo_obra_nivel_3,
                fecha_inicio_obra,
                fecha_finalizacion_real,
                codigo_entidad,
                nivel_gobierno,
                sector_entidad,
                codigo_unico,
                tipo_formato,
                n_obras_relacionadas_proyecto,
                departamento,
                provincia,
                distrito,
                fecha_aprobacion_expediente,
                monto_aprobacion_et,
                monto_aprobado_soles,
                ruc_1,
                razon_social_empresa_consorcio_1,
                monto_contrato_soles_1,
                ruc_2,
                razon_social_empresa_consorcio_2,
                monto_contrato_soles_2,
                flg_existe_paralizacion,
                CASE WHEN flg_existe_paralizacion THEN causal_paralizacion ELSE NULL END AS causal_paralizacion,
                CASE WHEN flg_existe_paralizacion THEN fecha_paralizacion ELSE NULL END AS fecha_paralizacion,
                estado_ejecucion,
                (NOT COALESCE(flg_existe_paralizacion, FALSE)
                    AND (causal_paralizacion IS NOT NULL OR fecha_paralizacion IS NOT NULL)) AS _regla3_aplicada
            FROM foto_bruta
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY codigo_infobras
                ORDER BY fecha_registro_avance DESC NULLS LAST,
                         ano_avance DESC NULLS LAST,
                         mes_avance DESC NULLS LAST
            ) = 1
        )
        SELECT
            d.*,
            CASE
                WHEN COALESCE(d.flg_existe_paralizacion, FALSE)
                    THEN COALESCE(m.categoria_paralizacion, '{cat_default_sql}')
                ELSE CAST(NULL AS VARCHAR)
            END AS categoria_paralizacion
        FROM dedup d
        LEFT JOIN _map_categoria_paralizacion m
               ON m.causal_norm = lower(trim(d.causal_paralizacion))
    """)
    con.execute("DROP TABLE IF EXISTS foto_bruta")

    filas_dedup, regla3_norm = con.execute(
        "SELECT count(*), COALESCE(SUM(CAST(_regla3_aplicada AS INTEGER)), 0) FROM foto_actual"
    ).fetchall()[0]
    return {
        "fecha_foto": fecha_foto,
        "filas_brutas": int(filas_brutas),
        "filas_deduplicadas": int(filas_dedup),
        "duplicados_eliminados": int(filas_brutas - filas_dedup),
        "regla3_normalizados": int(regla3_norm),
    }

## 6. Motores unificados SCD Tipo 1 (`TABLAS_SCD1`) y SCD Tipo 2 (`TABLAS_SCD2`)

In [ ]:
def actualizar_tabla_scd1(
    con: duckdb.DuckDBPyConnection,
    nombre_tabla: str,
    key_col: str,
    attr_cols: list[str],
    fuente_sql: str,
    fecha_foto: date,
) -> dict:
    """
    Actualización vectorizada SCD Tipo 1 (Hubs y sat_obra_ubigeo):
    - Claves existentes presentes en la foto: COALESCE(nuevo, actual) y fecha_foto = T.
    - Claves nuevas: inserta con first_seen_date = T y fecha_foto = T.
    - Claves ausentes en la foto: se conservan sin modificar.
    """
    f_iso = fecha_foto.isoformat()
    coalesce_exprs = ", ".join(f'COALESCE(f."{c}", h."{c}") AS "{c}"' for c in attr_cols)
    direct_exprs = ", ".join(f'f."{c}"' for c in attr_cols)

    con.execute(f"CREATE OR REPLACE TEMP TABLE _cand_scd1 AS {fuente_sql}")

    nuevas, actualizadas = con.execute(f"""
        SELECT
            count(*) FILTER (WHERE h."{key_col}" IS NULL),
            count(*) FILTER (WHERE h."{key_col}" IS NOT NULL)
        FROM _cand_scd1 f
        LEFT JOIN "{nombre_tabla}" h USING ("{key_col}")
    """).fetchall()[0]

    con.execute(f"""
        CREATE OR REPLACE TABLE "_next_{nombre_tabla}" AS
        SELECT
            h."{key_col}",
            {coalesce_exprs},
            h.first_seen_date,
            CASE WHEN f."{key_col}" IS NOT NULL THEN DATE '{f_iso}' ELSE h.fecha_foto END AS fecha_foto
        FROM "{nombre_tabla}" h
        LEFT JOIN _cand_scd1 f USING ("{key_col}")

        UNION ALL

        SELECT
            f."{key_col}",
            {direct_exprs},
            DATE '{f_iso}' AS first_seen_date,
            DATE '{f_iso}' AS fecha_foto
        FROM _cand_scd1 f
        LEFT JOIN "{nombre_tabla}" h USING ("{key_col}")
        WHERE h."{key_col}" IS NULL
    """)
    con.execute(f'DROP TABLE "{nombre_tabla}"')
    con.execute(f'ALTER TABLE "_next_{nombre_tabla}" RENAME TO "{nombre_tabla}"')
    con.execute("DROP TABLE IF EXISTS _cand_scd1")

    total = int(con.execute(f'SELECT count(*) FROM "{nombre_tabla}"').fetchall()[0][0])
    return {"nuevas": int(nuevas), "actualizadas": int(actualizadas), "total": total}


def actualizar_tabla_scd2(
    con: duckdb.DuckDBPyConnection,
    nombre_tabla: str,
    key_cols: list[str],
    cols_track: list[str],
    fuente_sql: str,
    fecha_foto: date,
) -> dict:
    """
    Actualización vectorizada SCD Tipo 2 con soporte de claves simples y compuestas:
    1. Conserva cerradas históricas (end_date IS NOT NULL).
    2. Sobre abiertas previas (end_date IS NULL):
       - Presente sin cambios (IS NOT DISTINCT FROM): actualiza fecha_foto = T.
       - Presente con cambios (IS DISTINCT FROM): cierra con end_date = T (preservando fecha_foto previa).
       - Ausente en la foto: mantiene end_date = NULL y fecha_foto previa (Regla 5).
    3. Inserta nuevas versiones (nuevas claves o cambios detectados) con start_date = T, end_date = NULL, fecha_foto = T.
    """
    f_iso = fecha_foto.isoformat()
    keys_plain = ", ".join(f'"{k}"' for k in key_cols)
    keys_c = ", ".join(f'c."{k}"' for k in key_cols)
    keys_t = ", ".join(f't."{k}"' for k in key_cols)
    using_keys = ", ".join(f'"{k}"' for k in key_cols)
    k0 = key_cols[0]

    cols_plain = ", ".join(f'"{c}"' for c in cols_track)
    cols_c = ", ".join(f'c."{c}"' for c in cols_track)
    cols_t = ", ".join(f't."{c}"' for c in cols_track)
    cond_igual = " AND ".join(f't."{c}" IS NOT DISTINCT FROM c."{c}"' for c in cols_track)

    con.execute(f"CREATE OR REPLACE TEMP TABLE _cand AS {fuente_sql}")

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE _abiertas AS
        SELECT {keys_plain}, {cols_plain}, start_date, end_date, fecha_foto
        FROM "{nombre_tabla}"
        WHERE end_date IS NULL
    """)

    nuevas, cambios, sin_cambio = con.execute(f"""
        SELECT
            count(*) FILTER (WHERE t."{k0}" IS NULL),
            count(*) FILTER (WHERE t."{k0}" IS NOT NULL AND NOT ({cond_igual})),
            count(*) FILTER (WHERE t."{k0}" IS NOT NULL AND ({cond_igual}))
        FROM _cand c
        LEFT JOIN _abiertas t USING ({using_keys})
    """).fetchall()[0]

    con.execute(f"""
        CREATE OR REPLACE TABLE "_next_{nombre_tabla}" AS
        SELECT {keys_plain}, {cols_plain}, start_date, end_date, fecha_foto
        FROM "{nombre_tabla}"
        WHERE end_date IS NOT NULL

        UNION ALL

        SELECT
            {keys_t},
            {cols_t},
            t.start_date,
            CASE
                WHEN c."{k0}" IS NOT NULL AND NOT ({cond_igual}) THEN DATE '{f_iso}'
                ELSE CAST(NULL AS DATE)
            END AS end_date,
            CASE
                WHEN c."{k0}" IS NOT NULL AND ({cond_igual}) THEN DATE '{f_iso}'
                ELSE t.fecha_foto
            END AS fecha_foto
        FROM _abiertas t
        LEFT JOIN _cand c USING ({using_keys})

        UNION ALL

        SELECT
            {keys_c},
            {cols_c},
            DATE '{f_iso}' AS start_date,
            CAST(NULL AS DATE) AS end_date,
            DATE '{f_iso}' AS fecha_foto
        FROM _cand c
        LEFT JOIN _abiertas t USING ({using_keys})
        WHERE t."{k0}" IS NULL OR NOT ({cond_igual})
    """)

    con.execute(f'DROP TABLE "{nombre_tabla}"')
    con.execute(f'ALTER TABLE "_next_{nombre_tabla}" RENAME TO "{nombre_tabla}"')
    con.execute("DROP TABLE IF EXISTS _cand")
    con.execute("DROP TABLE IF EXISTS _abiertas")

    total, abiertas, cerradas = con.execute(f"""
        SELECT
            count(*),
            count(*) FILTER (WHERE end_date IS NULL),
            count(*) FILTER (WHERE end_date IS NOT NULL)
        FROM "{nombre_tabla}"
    """).fetchall()[0]
    return {
        "nuevas": int(nuevas),
        "cambios": int(cambios),
        "sin_cambio": int(sin_cambio),
        "total": int(total),
        "abiertas": int(abiertas),
        "cerradas": int(cerradas),
    }

## 7. Escritura en `_work/`, validaciones bloqueantes (`S01`–`S06`) y promoción atómica

In [ ]:
TMP_ROOT = WORK_DIR / "_tmp"
OLD_ROOT = WORK_DIR / "_old"
ERROR_ROOT = WORK_DIR / "_error"


def escribir_work_parquet(con: duckdb.DuckDBPyConnection) -> dict[str, Path]:
    """Exporta las 13 tablas ordenadas a data/silver/_work/_tmp/<tabla>/part-00000.parquet."""
    if TMP_ROOT.exists():
        shutil.rmtree(TMP_ROOT)
    rutas = {}
    for tabla, spec in SCHEMAS_SILVER.items():
        carpeta = TMP_ROOT / tabla
        carpeta.mkdir(parents=True, exist_ok=True)
        archivo = carpeta / "part-00000.parquet"
        cols_expr = ", ".join(f'CAST("{c}" AS {t}) AS "{c}"' for c, t in spec["columns"])
        order_expr = ", ".join(f'"{c}"' for c in spec["order_by"])
        t0 = time.time()
        con.execute(
            f'COPY (SELECT {cols_expr} FROM "{tabla}" ORDER BY {order_expr}) '
            f"TO '{archivo.as_posix()}' "
            f"(FORMAT PARQUET, COMPRESSION {COMPRESION}, COMPRESSION_LEVEL {int(COMPRESION_NIVEL)}, ROW_GROUP_SIZE {int(ROW_GROUP_SIZE)})"
        ).fetchall()
        tam_mb = archivo.stat().st_size / 1024**2
        n_rows = con.execute(f'SELECT count(*) FROM "{tabla}"').fetchall()[0][0]
        log.info("[_work] %-28s escrito: %10s filas | %6.2f MB | %.2f s", tabla, f"{n_rows:,}", tam_mb, time.time() - t0)
        rutas[tabla] = archivo
    return rutas


def _scalar(con: duckdb.DuckDBPyConnection, sql: str):
    return con.execute(sql).fetchall()[0][0]


def validar_tablas_silver(con_mem: duckdb.DuckDBPyConnection, rutas_work: dict[str, Path]) -> pd.DataFrame:
    """Ejecuta validaciones bloqueantes S01-S06 sobre las 13 tablas Parquet escritas en _work/."""
    checks = []

    def chk(id_, tabla, descripcion, esperado, obtenido, ok):
        checks.append(
            {
                "id": id_,
                "tabla": tabla,
                "validacion": descripcion,
                "esperado": str(esperado),
                "obtenido": str(obtenido),
                "estado": "OK" if ok else "FALLA",
            }
        )

    con_v = duckdb.connect()
    try:
        for tabla, ruta in rutas_work.items():
            existe = ruta.exists() and ruta.stat().st_size > 0
            if not existe:
                chk("S06", tabla, "Parquet existe y no está vacío", "> 0 bytes", "no existe", False)
                return pd.DataFrame(checks)
            con_v.execute(f"CREATE VIEW \"{tabla}\" AS SELECT * FROM read_parquet('{ruta.as_posix()}')")

            # S06: esquema, tipos y conteo vs memoria
            desc = [(r[0], r[1]) for r in con_v.execute(f'DESCRIBE "{tabla}"').fetchall()]
            esperado_schema = SCHEMAS_SILVER[tabla]["columns"]
            n_pq = int(_scalar(con_v, f'SELECT count(*) FROM "{tabla}"'))
            n_mem = int(_scalar(con_mem, f'SELECT count(*) FROM "{tabla}"'))
            chk(
                "S06", tabla, "Esquema, tipos y conteo Parquet = memoria (> 0 filas)",
                f"{len(esperado_schema)} cols, {n_mem:,} filas",
                f"{len(desc)} cols, {n_pq:,} filas (schema_ok={desc == esperado_schema})",
                desc == esperado_schema and n_pq == n_mem and n_pq > 0,
            )

        # S01: Unicidad 100% de las 5 tablas SCD1
        for spec in TABLAS_SCD1:
            t = spec["nombre"]
            k = spec["key_col"]
            extra_zero = " OR trim(codigo_unico) = '0'" if t == "hub_proyecto" else ""
            n_tot, n_dist, n_inv = con_v.execute(
                f'SELECT count(*), count(DISTINCT "{k}"), '
                f'count(*) FILTER (WHERE "{k}" IS NULL OR trim("{k}") = \'\'{extra_zero}) '
                f'FROM "{t}"'
            ).fetchall()[0]
            chk(
                "S01", t, f"Unicidad 100% de {k} y sin nulos/inválidos",
                f"{n_tot:,} únicos, 0 inválidos", f"{n_dist:,} únicos, {n_inv} inválidos",
                n_tot == n_dist and n_inv == 0,
            )

        # S02 y S03 para las 8 tablas SCD2
        for spec in TABLAS_SCD2:
            t = spec["nombre"]
            keys_expr = ", ".join(f'"{k}"' for k in spec["key_cols"])

            # S02: a lo sumo 1 versión abierta por clave de seguimiento
            max_abiertas = int(_scalar(
                con_v,
                f'SELECT COALESCE(MAX(c), 0) FROM (SELECT {keys_expr}, count(*) AS c FROM "{t}" '
                f"WHERE end_date IS NULL GROUP BY {keys_expr})",
            ))
            chk("S02", t, f"A lo sumo 1 versión abierta por ({', '.join(spec['key_cols'])})", "<= 1", max_abiertas, max_abiertas <= 1)

            # S03: integridad temporal y ausencia de solapamientos
            n_err_temp = int(_scalar(
                con_v,
                f"""
                WITH w AS (
                    SELECT start_date, end_date, fecha_foto,
                           LEAD(start_date) OVER (PARTITION BY {keys_expr} ORDER BY start_date, end_date NULLS LAST) AS next_start
                    FROM "{t}"
                )
                SELECT count(*) FROM w
                WHERE start_date IS NULL
                   OR fecha_foto IS NULL
                   OR start_date > fecha_foto
                   OR (end_date IS NOT NULL AND end_date <= start_date)
                   OR (end_date IS NULL AND next_start IS NOT NULL)
                   OR (end_date IS NOT NULL AND next_start IS NOT NULL AND end_date > next_start)
                   OR (next_start IS NOT NULL AND next_start = start_date)
                """,
            ))
            chk("S03", t, "Integridad temporal (start_date <= fecha_foto, start < end, sin solapamientos)", 0, n_err_temp, n_err_temp == 0)

        # S04: Integridad referencial cruzada
        tablas_fk_obra = [
            "sat_obra_ubigeo",
            "link_obra_entidad",
            "link_obra_proyecto",
            "link_obra_contratista",
            "sat_obra_paralizacion",
            "sat_obra_estado_ejecucion",
            "sat_obra_expediente_tecnico",
            "sat_link_obra_contratista",
        ]
        for t in tablas_fk_obra:
            n_h = int(_scalar(
                con_v,
                f'SELECT count(*) FROM "{t}" s LEFT JOIN hub_obra_publica h USING (codigo_infobras) WHERE h.codigo_infobras IS NULL',
            ))
            chk("S04", t, "Integridad referencial codigo_infobras -> hub_obra_publica", 0, n_h, n_h == 0)

        n_h_ent = int(_scalar(
            con_v,
            "SELECT count(*) FROM link_obra_entidad l LEFT JOIN hub_entidad h USING (codigo_entidad) "
            "WHERE h.codigo_entidad IS NULL OR l.codigo_entidad IS NULL OR trim(l.codigo_entidad) = ''",
        ))
        chk("S04", "link_obra_entidad", "FK codigo_entidad -> hub_entidad", 0, n_h_ent, n_h_ent == 0)

        for t_proy in ["link_obra_proyecto", "sat_proyecto"]:
            n_h_proy = int(_scalar(
                con_v,
                f'SELECT count(*) FROM "{t_proy}" l LEFT JOIN hub_proyecto h USING (codigo_unico) '
                "WHERE h.codigo_unico IS NULL OR l.codigo_unico IS NULL OR trim(l.codigo_unico) IN ('', '0')",
            ))
            chk("S04", t_proy, "FK codigo_unico -> hub_proyecto (sin '0')", 0, n_h_proy, n_h_proy == 0)

        n_h_con = int(_scalar(
            con_v,
            "SELECT count(*) FROM link_obra_contratista l LEFT JOIN hub_contratista h USING (ruc) "
            "WHERE h.ruc IS NULL OR l.ruc IS NULL OR trim(l.ruc) = '' OR l.rol NOT IN ('EJECUCION', 'SUPERVISION')",
        ))
        chk("S04", "link_obra_contratista", "FK ruc -> hub_contratista y rol válido", 0, n_h_con, n_h_con == 0)

        n_h_slc = int(_scalar(
            con_v,
            """
            SELECT count(*)
            FROM sat_link_obra_contratista s
            LEFT JOIN (SELECT DISTINCT codigo_infobras, rol, ruc FROM link_obra_contratista) l
                   USING (codigo_infobras, rol, ruc)
            WHERE l.ruc IS NULL OR s.monto_contrato_soles IS NULL
            """,
        ))
        chk("S04", "sat_link_obra_contratista", "FK compuesta -> link_obra_contratista y monto no nulo", 0, n_h_slc, n_h_slc == 0)

        # S05: Regla 3 y categorización YAML en sat_obra_paralizacion
        cats_sql = ", ".join("'" + c.replace("'", "''") + "'" for c in CATEGORIAS_VALIDAS)
        n_regla3_err = int(_scalar(
            con_v,
            "SELECT count(*) FROM sat_obra_paralizacion "
            "WHERE flg_existe_paralizacion IS NULL "
            "   OR (flg_existe_paralizacion = FALSE AND (causal_paralizacion IS NOT NULL OR fecha_paralizacion IS NOT NULL OR categoria_paralizacion IS NOT NULL))",
        ))
        chk(
            "S05", "sat_obra_paralizacion",
            "Regla 3: flg_existe_paralizacion=FALSE => causal, categoria y fecha NULL",
            0, n_regla3_err, n_regla3_err == 0,
        )

        n_cat_err = int(_scalar(
            con_v,
            f"SELECT count(*) FROM sat_obra_paralizacion "
            f"WHERE flg_existe_paralizacion = TRUE AND (categoria_paralizacion IS NULL OR categoria_paralizacion NOT IN ({cats_sql}))",
        ))
        chk(
            "S05", "sat_obra_paralizacion",
            "Regla 8: flg_existe_paralizacion=TRUE => categoria_paralizacion dentro de YAML categories",
            0, n_cat_err, n_cat_err == 0,
        )
    finally:
        con_v.close()

    return pd.DataFrame(checks)


def promover_silver(rutas_work: dict[str, Path]) -> dict[str, Path]:
    """Promueve atómicamente las 13 tablas desde _work/_tmp hacia data/silver/<tabla>/ (con rollback)."""
    if OLD_ROOT.exists():
        shutil.rmtree(OLD_ROOT)
    OLD_ROOT.mkdir(parents=True, exist_ok=True)
    movidas_old, promovidas, finales = [], [], {}
    try:
        for tabla in rutas_work:
            destino = SILVER_DIR / tabla
            respaldo = OLD_ROOT / tabla
            if destino.exists():
                destino.rename(respaldo)
                movidas_old.append((destino, respaldo))
            origen = TMP_ROOT / tabla
            origen.rename(destino)
            promovidas.append(destino)
            finales[tabla] = destino / "part-00000.parquet"
    except Exception:
        for dest in promovidas:
            if dest.exists():
                shutil.rmtree(dest, ignore_errors=True)
        for dest, resp in movidas_old:
            if resp.exists() and not dest.exists():
                resp.rename(dest)
        raise
    shutil.rmtree(OLD_ROOT, ignore_errors=True)
    shutil.rmtree(TMP_ROOT, ignore_errors=True)
    return finales

## 8. Ejecución del proceso Silver

In [ ]:
con = duckdb.connect(database=":memory:")
t_total = time.time()
rutas_finales = {}
df_fotos = pd.DataFrame()
df_validacion = pd.DataFrame()
df_resumen_tablas = pd.DataFrame()

try:
    registrar_mapa_categoria_duckdb(con)
    todas_particiones = listar_particiones_stg(FECHA_MINIMA)
    assert todas_particiones, f"No se encontraron particiones válidas en {STG_DIR} desde {FECHA_MINIMA}"
    pendientes, watermark = inicializar_estado_duckdb(con, MODO, todas_particiones)
    log.info(
        "Particiones disponibles en Staging (>= %s): %d | Watermark previo: %s | Pendientes a procesar: %d",
        FECHA_MINIMA, len(todas_particiones), watermark, len(pendientes),
    )

    if not pendientes:
        log.info("No hay particiones nuevas posteriores a %s. No se modifica data/silver.", watermark)
    else:
        hist_fotos = []
        for idx, (fecha_foto, ruta_part) in enumerate(pendientes, start=1):
            t_f = time.time()
            st = cargar_foto_limpia(con, fecha_foto, ruta_part)
            fila_rep = dict(st)
            partes_log = []

            for spec in TABLAS_SCD1:
                st_s1 = actualizar_tabla_scd1(
                    con, spec["nombre"], spec["key_col"], spec["attr_cols"], spec["fuente_sql"], fecha_foto
                )
                fila_rep[f"{spec['nombre']}_nuevas"] = st_s1["nuevas"]
                fila_rep[f"{spec['nombre']}_upd"] = st_s1["actualizadas"]
                fila_rep[f"{spec['nombre']}_total"] = st_s1["total"]
                if spec["nombre"] == "hub_obra_publica":
                    partes_log.append(f"hub_obra(+{st_s1['nuevas']:,}/upd={st_s1['actualizadas']:,})")

            for spec in TABLAS_SCD2:
                st_t = actualizar_tabla_scd2(
                    con, spec["nombre"], spec["key_cols"], spec["cols_track"], spec["fuente_sql"], fecha_foto
                )
                fila_rep[f"{spec['nombre']}_nuevas"] = st_t["nuevas"]
                fila_rep[f"{spec['nombre']}_cambios"] = st_t["cambios"]
                fila_rep[f"{spec['nombre']}_total"] = st_t["total"]
                partes_log.append(f"{spec['nombre']}(+{st_t['nuevas']:,}/chg={st_t['cambios']:,})")

            con.execute("DROP TABLE IF EXISTS foto_actual")
            fila_rep["segundos"] = round(time.time() - t_f, 2)
            hist_fotos.append(fila_rep)
            log.info(
                "[%02d/%02d] %s | brutas=%s dedup=%s (-%s dups, R3=%s) | %s | %.2f s",
                idx, len(pendientes), fecha_foto,
                f"{st['filas_brutas']:,}", f"{st['filas_deduplicadas']:,}",
                st["duplicados_eliminados"], st["regla3_normalizados"],
                " | ".join(partes_log[:5]), fila_rep["segundos"],
            )

        df_fotos = pd.DataFrame(hist_fotos)
        df_fotos.to_csv(RESUMEN_FOTOS_PATH, index=False, encoding="utf-8-sig")
        log.info("Resumen por foto guardado: %s", RESUMEN_FOTOS_PATH)

        # Escritura temporal en _work/ y validaciones S01-S06
        rutas_work = escribir_work_parquet(con)
        df_validacion = validar_tablas_silver(con, rutas_work)
        for r in df_validacion.itertuples():
            (log.info if r.estado == "OK" else log.error)(
                "[VALIDACION] %s %-28s %s | esperado=%s | obtenido=%s -> %s",
                r.id, r.tabla, r.validacion, r.esperado, r.obtenido, r.estado,
            )

        n_fallas = int((df_validacion["estado"] == "FALLA").sum())
        if n_fallas > 0:
            err_dir = ERROR_ROOT / RUN_ID
            err_dir.mkdir(parents=True, exist_ok=True)
            if TMP_ROOT.exists():
                TMP_ROOT.rename(err_dir / "_tmp")
            raise RuntimeError(f"Se detectaron {n_fallas} falla(s) en validaciones Silver; artefactos movidos a {err_dir}")

        rutas_finales = promover_silver(rutas_work)
        resumen_tablas = []
        for tabla, ruta_f in rutas_finales.items():
            tam_mb = round(ruta_f.stat().st_size / 1024**2, 2)
            if tabla in SCD1_NOMBRES:
                tot, min_f, max_f = con.execute(
                    f'SELECT count(*), min(first_seen_date), max(fecha_foto) FROM "{tabla}"'
                ).fetchall()[0]
                ab, ce = tot, 0
            else:
                tot, ab, ce, min_f, max_f = con.execute(
                    f'SELECT count(*), count(*) FILTER (WHERE end_date IS NULL), count(*) FILTER (WHERE end_date IS NOT NULL), '
                    f'min(start_date), max(fecha_foto) FROM "{tabla}"'
                ).fetchall()[0]
            resumen_tablas.append({
                "tabla": tabla, "modelo": "SCD1" if tabla in SCD1_NOMBRES else "SCD2",
                "filas": int(tot), "abiertas": int(ab), "cerradas": int(ce),
                "min_fecha": min_f, "max_fecha_foto": max_f, "tamano_mb": tam_mb, "ruta": str(ruta_f),
            })
        df_resumen_tablas = pd.DataFrame(resumen_tablas)
        df_resumen_tablas.to_csv(RESUMEN_TABLAS_PATH, index=False, encoding="utf-8-sig")
        log.info("Promoción atómica completada en %s | Resumen guardado: %s", SILVER_DIR, RESUMEN_TABLAS_PATH)
except Exception as exc:
    log.error("ERROR EN CORRIDA SILVER: %s: %s\n%s", type(exc).__name__, exc, traceback.format_exc())
    if DETENER_ANTE_ERROR:
        raise
finally:
    con.close()
    gc.collect()
    log.info("FIN DE CORRIDA SILVER | run_id=%s | modo=%s | duración=%.2f s", RUN_ID, MODO, time.time() - t_total)
    log.info("=" * 100)

## 9. Resultados de validaciones (`S01`–`S06`), categorización y resumen de tablas

In [ ]:
if not df_validacion.empty:
    display(df_validacion)
if not df_resumen_tablas.empty:
    display(df_resumen_tablas.drop(columns=["ruta"]))
if not df_fotos.empty:
    display(df_fotos[["fecha_foto", "filas_brutas", "filas_deduplicadas", "duplicados_eliminados", "regla3_normalizados", "hub_obra_publica_nuevas", "hub_obra_publica_total", "sat_obra_paralizacion_total", "segundos"]])

## 10. Auditoría de trazabilidad y categorización sobre obras de control

In [ ]:
OBRAS_CONTROL = ["501650", "540677", "166922"]
if (SILVER_DIR / "hub_obra_publica" / "part-00000.parquet").exists():
    con_q = duckdb.connect()
    try:
        placeholders = ", ".join(f"'{c}'" for c in OBRAS_CONTROL)
        print("=== Distribución de categoria_paralizacion en sat_obra_paralizacion ===")
        display(con_q.execute(
            f"SELECT flg_existe_paralizacion, categoria_paralizacion, count(*) AS versiones "
            f"FROM read_parquet('{(SILVER_DIR / 'sat_obra_paralizacion' / '*.parquet').as_posix()}') "
            f"GROUP BY 1, 2 ORDER BY 1 DESC, 3 DESC"
        ).df())
        for t in ["hub_obra_publica", "sat_obra_ubigeo"]:
            print(f"=== {t} ===")
            display(con_q.execute(
                f"SELECT * FROM read_parquet('{(SILVER_DIR / t / '*.parquet').as_posix()}') "
                f"WHERE codigo_infobras IN ({placeholders}) ORDER BY codigo_infobras"
            ).df())
        for t in [
            "sat_obra_paralizacion",
            "sat_obra_estado_ejecucion",
            "sat_obra_expediente_tecnico",
            "link_obra_contratista",
            "sat_link_obra_contratista",
            "link_obra_proyecto",
            "link_obra_entidad",
        ]:
            print(f"=== {t} ===")
            display(con_q.execute(
                f"SELECT * FROM read_parquet('{(SILVER_DIR / t / '*.parquet').as_posix()}') "
                f"WHERE codigo_infobras IN ({placeholders}) ORDER BY codigo_infobras, start_date"
            ).df())
    finally:
        con_q.close()
else:
    print("Aún no existen tablas en", SILVER_DIR)